In [2]:
import sys
from pathlib import Path
import pandas as pd

print("Python:", sys.executable)
print("작업 폴더:", Path.cwd())
print("pandas:", pd.__version__)

for name in [
    "menugen_menu_ingredients.csv",
    "db_menu.csv",
    "db_ingredient.csv",
]:
    print(name, (Path("../dataset") / name).exists())

Python: c:\Python312\python.exe
작업 폴더: d:\HUMAN-meal-planner-ai\ai-server\training
pandas: 2.3.3
menugen_menu_ingredients.csv True
db_menu.csv True
db_ingredient.csv True


In [3]:
from pathlib import Path
import pandas as pd

dataset_dir = Path("../dataset")

recipes = pd.read_csv(
    dataset_dir / "menugen_menu_ingredients.csv",
    encoding="utf-8-sig",
    dtype={"menu_fd_Code": "string", "ingredient_food_Code": "string"},
)
menus = pd.read_csv(
    dataset_dir / "db_menu.csv",
    encoding="utf-8-sig",
    dtype={"menu_code": "string"},
)
ingredients = pd.read_csv(
    dataset_dir / "db_ingredient.csv",
    encoding="utf-8-sig",
    dtype={"ingredient_code": "string"},
)

for frame, columns in [
    (recipes, ["menu_fd_Code", "ingredient_food_Code"]),
    (menus, ["menu_code"]),
    (ingredients, ["ingredient_code"]),
]:
    for column in columns:
        frame[column] = frame[column].str.strip()

merged = recipes.merge(
    menus[["menu_id", "menu_code"]],
    left_on="menu_fd_Code",
    right_on="menu_code",
    how="left",
    validate="many_to_one",
)
merged = merged.merge(
    ingredients[["ingredient_id", "ingredient_code", "name", "standard_unit"]],
    left_on="ingredient_food_Code",
    right_on="ingredient_code",
    how="left",
    validate="many_to_one",
)

merged["raw_quantity"] = pd.to_numeric(
    merged["ingredient_food_Wgh"],
    errors="coerce",
)

reasons = pd.Series("", index=merged.index, dtype="string")
reasons.loc[merged["menu_id"].isna()] += "MENU_NOT_FOUND;"
reasons.loc[merged["ingredient_id"].isna()] += "INGREDIENT_NOT_FOUND;"
bad_quantity = merged["raw_quantity"].isna() | (merged["raw_quantity"] <= 0)
reasons.loc[bad_quantity] += "INVALID_QUANTITY;"
merged["error_reason"] = reasons

merged.loc[reasons.eq("")].to_csv(
    dataset_dir / "menu_ingredient_matched_for_review.csv",
    index=False,
    encoding="utf-8-sig",
)
merged.loc[~reasons.eq("")].to_csv(
    dataset_dir / "menu_ingredient_errors.csv",
    index=False,
    encoding="utf-8-sig",
)

print("매칭 성공:", int(reasons.eq("").sum()))
print("오류:", int((~reasons.eq("")).sum()))

매칭 성공: 22560
오류: 22


In [4]:
errors = pd.read_csv(
    dataset_dir / "menu_ingredient_errors.csv",
    encoding="utf-8-sig",
)

print(errors["error_reason"].value_counts())
display(errors[
    [
        "menu_fd_Code",
        "menu_fd_Nm",
        "ingredient_food_Code",
        "ingredient_food_Nm",
        "ingredient_food_Wgh",
        "error_reason",
    ]
])

error_reason
INVALID_QUANTITY;    22
Name: count, dtype: int64


,menu_fd_Code,menu_fd_Nm,ingredient_food_Code,ingredient_food_Nm,ingredient_food_Wgh,error_reason
0,D012064,오곡연잎밥,F03015,연잎밥,0.0,INVALID_QUANTITY;
1,D013024,단호박약식,F03083,"치자꽃, 생것",0.0,INVALID_QUANTITY;
2,D014003,비빔밥(고추장),F02712,콩기름,0.0,INVALID_QUANTITY;
3,D014004,"비빔밥(고추장, 돼지고기)",F02712,콩기름,0.0,INVALID_QUANTITY;
4,D014006,산채비빔밥(고추장),F02712,콩기름,0.0,INVALID_QUANTITY;
5,D014012,해초비빔밥(고추장),F02825,"간장, 개량, 양조",0.0,INVALID_QUANTITY;
6,D014012,해초비빔밥(고추장),F02876,"소금, 천일염, 가는소금",0.0,INVALID_QUANTITY;
7,D014019,비빔밥,F02827,"간장, 재래",0.0,INVALID_QUANTITY;
8,D014019,비빔밥,F00394,"설탕, 백설탕",0.0,INVALID_QUANTITY;
9,D014019,비빔밥,F02876,"소금, 천일염, 가는소금",0.0,INVALID_QUANTITY;


In [5]:
matched = pd.read_csv(
    dataset_dir / "menu_ingredient_matched_for_review.csv",
    encoding="utf-8-sig",
)

print(matched["standard_unit"].value_counts(dropna=False))
display(matched[["ingredient_food_Wgh", "raw_quantity", "standard_unit"]].head(10))

standard_unit
g    22560
Name: count, dtype: int64


,ingredient_food_Wgh,raw_quantity,standard_unit
0,440.0,440.0,g
1,210.0,210.0,g
2,210.0,210.0,g
3,210.0,210.0,g
4,160.0,160.0,g
5,100.0,100.0,g
6,70.0,70.0,g
7,20.0,20.0,g
8,90.0,90.0,g
9,10.0,10.0,g


In [6]:
review = matched[
    [
        "menu_id",
        "menu_code",
        "menu_fd_Nm",
        "ingredient_id",
        "ingredient_code",
        "ingredient_food_Nm",
        "raw_quantity",
        "standard_unit",
    ]
].copy()

review["is_primary_candidate"] = review["raw_quantity"].eq(
    review.groupby("menu_id")["raw_quantity"].transform("max")
)

review.sort_values(
    ["menu_code", "raw_quantity"],
    ascending=[True, False],
).to_csv(
    dataset_dir / "menu_ingredient_primary_review.csv",
    index=False,
    encoding="utf-8-sig",
)

In [7]:
review = pd.read_csv(
    dataset_dir / "menu_ingredient_primary_review.csv",
    encoding="utf-8-sig",
)

review["is_primary"] = (
    review["is_primary_candidate"]
    .astype("string")
    .str.lower()
    .eq("true")
)

provisional = review[
    ["menu_id", "ingredient_id", "raw_quantity", "is_primary"]
].rename(columns={"raw_quantity": "quantity"})

provisional.to_csv(
    dataset_dir / "menu_ingredient_provisional.csv",
    index=False,
    encoding="utf-8-sig",
)